# 3. Outlier Analysis

This notebook investigates unusual annual spending patterns before clustering.

The analysis combines univariate IQR rules with multidimensional methods.
Flagged customers are reviewed before deciding whether they should be
excluded from model fitting.

High spending alone is not treated as evidence of a data error. All customer
identifiers are retained so that every customer can receive a final segment.

## Import and Data 


In [1]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

In [2]:
import numpy as np
import pandas as pd

from utils.outlier_functions import (
    calculate_iqr_bounds,
    flag_iqr_outliers,
    plot_spending_distribution
)


In [3]:
customer_info = pd.read_csv(r"C:\Users\nhatp\OneDrive - NOVAIMS\gits\customer-segmentation\data\preprocessed\customer_info_clean.csv")

print(f"Customers: {customer_info.shape[0]:,}")
print(f"Features: {customer_info.shape[1]}")

Customers: 34,060
Features: 25


## Spending Feature Selection



Outlier detection initially uses the ten annual spending categories.

Customer identifiers, demographics, location, and categorical indicators
are retained for profiling but excluded from the spending-distance analysis.

Total spending is also excluded from the detector because it is derived
from the category spending variables.

In [6]:
spend_columns = [column for column in customer_info.columns if column.startswith('spend_per_year_')]
spend_columns

['spend_per_year_groceries',
 'spend_per_year_electronics',
 'spend_per_year_vegetables',
 'spend_per_year_nonalcohol_drinks',
 'spend_per_year_alcohol_drinks',
 'spend_per_year_meat',
 'spend_per_year_fish',
 'spend_per_year_hygiene',
 'spend_per_year_petfood',
 'spend_per_year_videogames']

In [7]:
X = customer_info[spend_columns].copy()

assert customer_info["customer_id"].is_unique
assert X.notna().all().all()
assert np.isfinite(X.to_numpy()).all()
assert X.ge(0).all().all()

In [8]:
X.describe(
    percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]
).T.round(2)

,count,mean,std,min,25%,50%,75%,95%,99%,max
spend_per_year_groceries,34060.0,2927.74,4196.25,0.00,941.98,1763.54,3328.31,9226.90,19330.23,88706.0
spend_per_year_electronics,34060.0,518.65,854.51,0.00,99.62,267.94,544.39,1981.02,3983.73,26214.0
spend_per_year_vegetables,34060.0,96.53,149.79,0.00,22.40,49.62,122.88,287.67,768.81,2602.0
spend_per_year_nonalcohol_drinks,34060.0,60.31,71.62,0.00,26.64,43.25,68.00,164.00,374.61,1547.0
spend_per_year_alcohol_drinks,34060.0,110.73,153.38,0.06,33.93,70.44,133.17,318.20,784.24,2908.0
spend_per_year_meat,34060.0,117.12,134.11,0.00,54.80,89.12,139.14,302.25,720.70,2208.0
spend_per_year_fish,34060.0,105.43,142.89,0.00,28.79,72.32,129.00,310.75,684.70,2886.0
spend_per_year_hygiene,34060.0,89.38,115.67,0.00,34.82,63.50,103.83,242.50,550.82,2734.0
spend_per_year_petfood,34060.0,41.70,45.64,0.00,19.42,30.00,47.50,110.75,239.50,796.0
spend_per_year_videogames,34060.0,54.34,72.85,0.00,17.17,29.50,59.44,191.01,342.60,1872.0


## Univariate Outlier Analysis



IQR bounds are calculated separately for each annual spending category.

A multiplier of 10 is used as a conservative starting point, following the
previous analysis. This is a screening choice, not a validated definition
of erroneous spending.

Customers are flagged when their spending falls outside the bounds in
at least one category. They are not removed at this stage.

In [9]:
iqr_multiplier = 10

iqr_bounds = calculate_iqr_bounds(
    customer_info,
    spend_columns,
    multiplier=iqr_multiplier,
)

iqr_flags = flag_iqr_outliers(
    customer_info,
    iqr_bounds,
)

In [10]:
iqr_summary = iqr_bounds.copy()

iqr_summary["customers_flagged"] = iqr_flags.sum()
iqr_summary["percentage_flagged"] = iqr_flags.mean() * 100

iqr_summary.sort_values(
    "customers_flagged",
    ascending=False,
).round(2)

,q1,q3,iqr,lower_bound,upper_bound,customers_flagged,percentage_flagged
spend_per_year_nonalcohol_drinks,26.64,68.00,41.36,-387.00,481.64,188,0.55
spend_per_year_electronics,99.62,544.39,444.77,-4348.12,4992.13,170,0.50
spend_per_year_alcohol_drinks,33.93,133.17,99.24,-958.47,1125.56,167,0.49
spend_per_year_hygiene,34.82,103.83,69.02,-655.33,793.98,163,0.48
spend_per_year_groceries,941.98,3328.31,2386.34,-22921.38,27191.66,160,0.47
spend_per_year_petfood,19.42,47.50,28.08,-261.37,328.29,159,0.47
spend_per_year_meat,54.80,139.14,84.34,-788.56,982.49,142,0.42
spend_per_year_videogames,17.17,59.44,42.28,-405.61,482.22,105,0.31
spend_per_year_fish,28.79,129.00,100.21,-973.28,1131.08,104,0.31
spend_per_year_vegetables,22.40,122.88,100.48,-982.37,1127.65,101,0.30


In [11]:
outlier_flags = customer_info[["customer_id"]].copy()

outlier_flags["iqr_outlier"] = iqr_flags.any(axis=1)
outlier_flags["iqr_features_flagged"] = iqr_flags.sum(axis=1)

print(
    f"Customers flagged: "
    f"{outlier_flags['iqr_outlier'].sum():,}"
)

print(
    f"Percentage flagged: "
    f"{outlier_flags['iqr_outlier'].mean():.2%}"
)

Customers flagged: 614
Percentage flagged: 1.80%


In [14]:
plot_spending_distribution(
    customer_info,
    "spend_per_year_groceries",
    iqr_bounds,
).show()

In [13]:
plot_spending_distribution(
    customer_info,
    "spend_per_year_vegetables",
    iqr_bounds,
).show()

In [15]:
sensitivity_results = []

for multiplier in [3, 5, 10]:
    bounds = calculate_iqr_bounds(
        customer_info,
        spend_columns,
        multiplier=multiplier,
    )

    flags = flag_iqr_outliers(customer_info, bounds)
    customer_flags = flags.any(axis=1)

    sensitivity_results.append({
        "iqr_multiplier": multiplier,
        "customers_flagged": customer_flags.sum(),
        "percentage_flagged": customer_flags.mean() * 100,
    })

pd.DataFrame(sensitivity_results).round(2)

,iqr_multiplier,customers_flagged,percentage_flagged
0,3,4420,12.98
1,5,2160,6.34
2,10,614,1.80


In [17]:
review_columns = [
    "customer_id",
    "age",
    "customer_lifetime",
    "pct_prod_in_promotion",
] + spend_columns

flagged_customers = customer_info.loc[
    outlier_flags["iqr_outlier"],
    review_columns,
].copy()

flagged_customers["iqr_features_flagged"] = (
    outlier_flags.loc[
        outlier_flags["iqr_outlier"],
        "iqr_features_flagged",
    ]
)

flagged_customers.sort_values(
    "iqr_features_flagged",
    ascending=False,
).head(15)

,customer_id,age,customer_lifetime,pct_prod_in_promotion,spend_per_year_groceries,spend_per_year_electronics,spend_per_year_vegetables,spend_per_year_nonalcohol_drinks,spend_per_year_alcohol_drinks,spend_per_year_meat,spend_per_year_fish,spend_per_year_hygiene,spend_per_year_petfood,spend_per_year_videogames,iqr_features_flagged
15898,8643,25,1.0,0.031999,45292.0,5821.0,1608.000000,690.0,1704.0,1255.0,1403.0,1405.0,393.0,527.0,10
21494,34287,25,1.0,0.023356,60693.0,9201.0,1491.000000,545.0,1669.0,1433.0,2358.0,1337.0,116.0,625.0,9
16704,39014,25,1.0,0.142903,72821.0,1799.0,2213.000000,520.0,1543.0,1845.0,1755.0,913.0,546.0,703.0,9
17777,22805,25,1.0,0.012864,68766.0,2818.0,2499.000000,490.0,1576.0,1157.0,1538.0,1144.0,437.0,660.0,9
21467,5715,25,1.0,0.017426,80809.0,8271.0,1658.000000,761.0,1772.0,1011.0,2724.0,800.0,263.0,501.0,9
24682,2939,25,1.0,0.145834,66933.0,5594.0,2120.000000,555.0,1682.0,1563.0,928.0,1103.0,553.0,631.0,9
28295,24630,25,1.0,0.202459,37433.0,5371.0,908.000000,414.0,1385.0,1335.0,1373.0,1052.0,577.0,513.0,8
12363,12680,25,1.0,0.052008,77707.0,4479.0,2166.000000,505.0,1429.0,1176.0,1774.0,1449.0,169.0,498.0,8
14272,20985,25,1.0,0.111667,10946.0,6765.0,610.000000,623.0,1732.0,1379.0,1240.0,1205.0,420.0,851.0,8
21456,11399,25,1.0,0.044998,60890.0,10424.0,711.000000,508.0,1346.0,947.0,1979.0,1175.0,361.0,565.0,8


## Relationship Between Outlier Flags and Customer Tenure

Annual spending depends on customer tenure. Customers with zero tenure
require estimated spending values, while customers with one year of tenure
retain their recorded lifetime spending as their annual average.

The following checks examine whether IQR flags are concentrated among
short-tenure customers before any removal decision is made.

In [18]:
outlier_review = customer_info.merge(
    outlier_flags,
    on="customer_id",
    how="left",
    validate="one_to_one",
)

tenure_summary = (
    outlier_review
    .groupby("customer_lifetime")
    .agg(
        customers=("customer_id", "size"),
        customers_flagged=("iqr_outlier", "sum"),
        flag_rate=("iqr_outlier", "mean"),
        median_features_flagged=("iqr_features_flagged", "median"),
    )
)

tenure_summary["flag_rate"] *= 100

tenure_summary.round(2)

,customers,customers_flagged,flag_rate,median_features_flagged
customer_lifetime,,,,
0.0,345,89,25.80,0.0
1.0,310,286,92.26,2.0
2.0,401,145,36.16,0.0
3.0,365,27,7.40,0.0
4.0,3163,62,1.96,0.0
5.0,1457,5,0.34,0.0
6.0,1738,0,0.00,0.0
7.0,2060,0,0.00,0.0
8.0,2369,0,0.00,0.0


In [19]:
age_summary = (
    outlier_review
    .groupby("age")
    .agg(
        customers=("customer_id", "size"),
        customers_flagged=("iqr_outlier", "sum"),
        flag_rate=("iqr_outlier", "mean"),
    )
)

age_summary["flag_rate"] *= 100

age_summary.sort_values(
    "flag_rate",
    ascending=False,
).round(2).head(15)

,customers,customers_flagged,flag_rate
age,,,
23,1,1,100.00
25,564,285,50.53
26,539,144,26.72
24,588,88,14.97
27,561,28,4.99
52,526,4,0.76
44,512,3,0.59
54,868,5,0.58
36,521,3,0.58


In [20]:
outlier_review.groupby(
    outlier_review["customer_lifetime"].eq(0).rename("zero_tenure")
)[spend_columns].median().T.round(2)

zero_tenure,False,True
spend_per_year_groceries,1737.86,13210.90
spend_per_year_electronics,264.44,1878.50
spend_per_year_vegetables,48.78,760.79
spend_per_year_nonalcohol_drinks,42.87,246.73
spend_per_year_alcohol_drinks,69.47,487.07
spend_per_year_meat,88.43,470.63
spend_per_year_fish,71.47,397.10
spend_per_year_hygiene,62.83,411.90
spend_per_year_petfood,29.78,164.50
spend_per_year_videogames,29.20,203.40
